In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image, ImageStat


def mean_std_rgb_hsv_lab_size(image_path: str) -> np.ndarray:
    """Extract handcrafted image statistics.

    Returns a 1‑D array containing:
    - mean & std for RGB (3 + 3)
    - green‑to‑red & green‑to‑blue ratios (2)
    - mean & std for HSV (3 + 3)
    - mean & std for LAB (3 + 3)
    - mean & std for grayscale (2)
    - width, height, aspect ratio, area (4)

    Total length = 26.
    """
    with Image.open(image_path).convert("RGB") as img:
        stat_rgb = ImageStat.Stat(img)
        mean_rgb = np.array(stat_rgb.mean, dtype=np.float32) / 255.0
        std_rgb = np.array(stat_rgb.stddev, dtype=np.float32) / 255.0

        red, green, blue = mean_rgb
        ratio_g_r = green / (red + 1e-6)
        ratio_g_b = green / (blue + 1e-6)

        img_hsv = img.convert("HSV")
        stat_hsv = ImageStat.Stat(img_hsv)
        mean_hsv = np.array(stat_hsv.mean, dtype=np.float32) / 255.0
        std_hsv = np.array(stat_hsv.stddev, dtype=np.float32) / 255.0

        img_lab = img.convert("LAB")
        stat_lab = ImageStat.Stat(img_lab)
        mean_lab = np.array(stat_lab.mean, dtype=np.float32) / 255.0
        std_lab = np.array(stat_lab.stddev, dtype=np.float32) / 255.0

        img_gray = img.convert("L")
        stat_gray = ImageStat.Stat(img_gray)
        mean_gray = float(np.array(stat_gray.mean, dtype=np.float32) / 255.0)
        std_gray = float(np.array(stat_gray.stddev, dtype=np.float32) / 255.0)

        width, height = img.size
        width_f = width / 1000.0
        height_f = height / 1000.0
        aspect = width_f / height_f if height_f != 0 else 0.0
        area = (width * height) / 1e6  # megapixels

        return np.concatenate(
            [
                mean_rgb,
                std_rgb,
                np.array([ratio_g_r, ratio_g_b], dtype=np.float32),
                mean_hsv,
                std_hsv,
                mean_lab,
                std_lab,
                np.array([mean_gray, std_gray], dtype=np.float32),
                np.array([width_f, height_f, aspect, area], dtype=np.float32),
            ]
        )




In [2]:
from concurrent.futures import ThreadPoolExecutor
import multiprocessing

train_csv_path = "/kaggle/input/cassava-leaf-disease-classification/train.csv"
train_image_dir = "/kaggle/input/cassava-leaf-disease-classification/train_images"

if not os.path.exists(train_csv_path):
    raise FileNotFoundError(f"train.csv not found at {train_csv_path}")

train_df = pd.read_csv(train_csv_path)

train_paths = []
train_labels = []
for img_id, label in zip(train_df["image_id"], train_df["label"]):
    img_path = os.path.join(train_image_dir, img_id)
    if os.path.exists(img_path):
        train_paths.append(img_path)
        train_labels.append(label)

print("Extracting features from training images (parallel with threads)...")
max_workers = min(32, multiprocessing.cpu_count() + 4)
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    train_features_list = list(
        executor.map(mean_std_rgb_hsv_lab_size, train_paths, chunksize=32)
    )

train_features = np.stack(train_features_list)  # shape (n_samples, 26)
train_labels = np.asarray(train_labels)




Extracting features from training images (parallel with threads)...


/tmp/ipykernel_55/1752071654.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  mean_gray = float(np.array(stat_gray.mean, dtype=np.float32) / 255.0)
/tmp/ipykernel_55/1752071654.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  std_gray = float(np.array(stat_gray.stddev, dtype=np.float32) / 255.0)


In [3]:
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier

# ExtraTrees model – retained from the original pipeline
rf_model = ExtraTreesClassifier(
    n_estimators=2500,
    criterion="gini",
    max_depth=None,
    min_samples_split=2,
    max_features=0.8,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
)
rf_model.fit(train_features, train_labels)

# Additional Gradient‑Boosted model to capture complementary patterns
hgb_model = HistGradientBoostingClassifier(
    max_iter=500,
    learning_rate=0.05,
    random_state=42,
)
hgb_model.fit(train_features, train_labels)




HistGradientBoostingClassifier(learning_rate=0.05, max_iter=500,
                               random_state=42)

In [4]:
test_dir = "/kaggle/input/cassava-leaf-disease-classification/test_images"
if not os.path.isdir(test_dir):
    raise NotADirectoryError(f"Test image directory not found: {test_dir}")

test_images = sorted(
    [f for f in os.listdir(test_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
)

test_paths = [os.path.join(test_dir, img_name) for img_name in test_images]
image_ids = test_images  # preserve ordering for submission

print("Extracting features from test images (parallel with threads)...")
max_workers = min(32, multiprocessing.cpu_count() + 4)
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    test_features_list = list(
        executor.map(mean_std_rgb_hsv_lab_size, test_paths, chunksize=32)
    )

test_features = np.stack(test_features_list)  # shape (n_test, 26)




Extracting features from test images (parallel with threads)...


/tmp/ipykernel_55/1752071654.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  mean_gray = float(np.array(stat_gray.mean, dtype=np.float32) / 255.0)
/tmp/ipykernel_55/1752071654.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  std_gray = float(np.array(stat_gray.stddev, dtype=np.float32) / 255.0)


In [5]:
if test_features.shape[0] == 0:
    raise RuntimeError("No test images were processed; cannot create predictions.")

# Obtain class probabilities from both models
proba_rf = rf_model.predict_proba(test_features)
proba_hgb = hgb_model.predict_proba(test_features)

# Average the probabilities to get a consensus prediction
avg_proba = (proba_rf + proba_hgb) / 2.0
prediction = np.argmax(avg_proba, axis=1)

submission = pd.DataFrame({"image_id": image_ids, "label": prediction.astype(int)})
submission = submission.sort_values("image_id").reset_index(drop=True)

submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission file written to {submission_path}")

print("\nSubmission preview:")
print(submission.head())

Submission file written to submission.csv

Submission preview:
         image_id  label
0  1234294272.jpg      3
1  1234332763.jpg      3
2  1234375577.jpg      3
3  1234555380.jpg      3
4  1234571117.jpg      3
